# Configure ADLS Gen2 Access in Databricks

Configures Spark to access the `kalshi-data` container via OAuth. Uses **direct ABFSS paths** (no mount) for workspaces where DBFS mounts are disabled.

**Usage:** After running this notebook, use `KALSHI_DATA_PATH` in your pipelines:
- `spark.read.format("delta").load(f"{KALSHI_DATA_PATH}/bronze/markets")`
- `df.write.format("delta").save(f"{KALSHI_DATA_PATH}/bronze/raw")`

**Prerequisites:**
1. Secret scope `kalshi-secrets` linked to Azure Key Vault
2. Key Vault secret `sp-client-secret` (service principal client secret)
3. Service principal has **Storage Blob Data Contributor** on the storage account

In [ ]:
# Config - update STORAGE_ACCOUNT from: az deployment group show -g rg-kalshi-pipeline -n kalshi-deploy --query properties.outputs.storageAccountName.value -o tsv
STORAGE_ACCOUNT = ""  # e.g. stkalshiogihujuict7io from deployment outputs
TENANT_ID = "b4b203c1-e6ed-4319-a1aa-80694c9ce7e9"
CLIENT_ID = "5e532278-857e-493b-9185-ea6b714d1e42"  # sp-kalshi-databricks

In [ ]:
container = "kalshi-data"

# Strip whitespace (avoids "Illegal character" from stray spaces)
STORAGE_ACCOUNT = STORAGE_ACCOUNT.strip()
TENANT_ID = TENANT_ID.strip()
CLIENT_ID = CLIENT_ID.strip()

if not STORAGE_ACCOUNT or not TENANT_ID or not CLIENT_ID:
    raise ValueError("Set STORAGE_ACCOUNT, TENANT_ID, and CLIENT_ID in the config cell above")

client_secret = dbutils.secrets.get(scope="kalshi-secrets", key="sp-client-secret")

# Configure Spark for direct ABFSS access (no mount - works when DBFS mounts are disabled)
spark.conf.set(f"fs.azure.account.auth.type.{STORAGE_ACCOUNT}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{STORAGE_ACCOUNT}.dfs.core.windows.net",
               "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{STORAGE_ACCOUNT}.dfs.core.windows.net", CLIENT_ID)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{STORAGE_ACCOUNT}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{STORAGE_ACCOUNT}.dfs.core.windows.net",
               f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/token")

# Use this path in your pipelines (e.g. spark.read.format("delta").load(f"{KALSHI_DATA_PATH}/bronze/markets"))
KALSHI_DATA_PATH = f"abfss://{container}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
print(f"Configured. Use path: {KALSHI_DATA_PATH}")

In [ ]:
# Verify - list root of container
dbutils.fs.ls(KALSHI_DATA_PATH)